# Глава 7: Подготовка данных
**Создайте свою первую LLM — Глава 7: Подготовка данных**

Этот блокнот демонстрирует шаги по подготовке данных: очистка, дедупликация, разделение, чанкинг и сохранение в JSONL с быстрой статистикой.
Небольшие игрушечные данные встроены, никаких внешних файлов не требуется.

In [ ]:
# ===== ИМПОРТЫ =====
import re           # Регулярные выражения для поиска по шаблонам в тексте
import json         # Чтение/запись JSON и JSONL файлов
import hashlib      # Создание отпечатков для обнаружения дубликатов
import unicodedata  # Нормализация Unicode (обработка специальных символов)
from collections import Counter  # Подсчет частоты слов
import random       # Перемешивание данных для разделения train/val/test

print('Настройка завершена')

## Краткое руководство по инструментам Python

Этот блокнот использует несколько инструментов Python. Вот краткое руководство:

**Регулярные выражения (regex):** Поиск и замена на основе шаблонов в тексте
- `re.sub(pattern, replacement, text)` — Найти все совпадения `pattern` и заменить
- Шаблон `<[^>]+>` означает: найти `<`, затем любые символы, которые не являются `>`, затем `>` → соответствует HTML-тегам

**Хеширование:** Создание уникального "отпечатка" для любого текста
- Одинаковый текст → одинаковый хеш (всегда). Разный текст → разный хеш (почти всегда)
- Полезно для обнаружения дубликатов без сравнения целых документов
- `hashlib.sha1(text.encode()).hexdigest()` → 40-символьный отпечаток

**JSON/JSONL:** Форматы данных для хранения структурированных данных
- **JSON:** Один большой файл со всеми данными (необходимо загружать весь файл в память)
- **JSONL:** Одна JSON-запись на строку (можно читать построчно = эффективное использование памяти)

**Множества (Sets):** Коллекции без дубликатов, быстрая проверка "есть ли X в этом множестве?"
- `seen = set()` затем `seen.add(item)` и `item in seen`

**Аннотации типов** (`: str`, `: float = 0.8`): Документация для людей (Python их игнорирует)
- `text: str` означает "text должен быть строкой"
- `train_p: float = 0.8` означает "train_p должен быть числом с плавающей точкой, по умолчанию 0.8"

## Примеры текстов
Несколько игрушечных параграфов для имитации небольшого корпуса.

In [ ]:
raw_docs = [
    "THE TIME MACHINE — CHAPTER I\n\nThis   is   a    sample text… with   odd spacing, smart “quotes”, and tabs\t.",
    "AI systems learn from examples. Data quality shapes model quality.",
    "The key to machine learning is data; the secret to building AI is understanding.",
]
print('Документов:', len(raw_docs))


## Очистка и нормализация

Удаление HTML, управляющих символов, сжатие пробелов, нормализация кавычек.

In [ ]:
def clean_text(text: str) -> str:
    """Очистка и нормализация текста для обучения LLM."""
    # Нормализация Unicode: ﬁ → fi, ｆｕｌｌ → full, и т.д.
    # NFKC = Декомпозиция совместимости + Каноническая композиция
    text = unicodedata.normalize("NFKC", text)
    
    # Удаление HTML-тегов: <p>, <div>, <span class="foo">, и т.д.
    # Шаблон: < затем любые символы, которые не >, затем >
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Замена переводов строк и табуляций на пробелы
    text = re.sub(r"[\n\t]", " ", text)
    
    # Сжатие нескольких пробелов в один, удаление начальных/конечных пробелов
    text = re.sub(r"\s+", " ", text).strip()
    
    # Нормализация умных кавычек в обычные кавычки
    text = text.replace("“", '"').replace("”", '"')
    
    return text

cleaned_docs = [clean_text(d) for d in raw_docs]
for i, d in enumerate(cleaned_docs):
    print(f"Очищено {i}: {d[:80]}...")

## Дедупликация

Хеширование параграфов, удаление повторов.

In [ ]:
# Создание "отпечатка" для текста с использованием SHA1-хеша
# Одинаковый текст → одинаковый отпечаток (всегда)
# Разный текст → разный отпечаток (с подавляющей вероятностью)
def hash_chunk(text: str) -> str:
    # .encode("utf-8") преобразует строку в байты (требуется для hashlib)
    # .hexdigest() возвращает хеш в виде 40-символьной строки
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

# Демонстрация хеширования
sample = "The cat sat on the mat."
print(f"Текст: '{sample}'")
print(f"Хеш: {hash_chunk(sample)}")
print(f"Одинаковый текст, одинаковый хеш: {hash_chunk(sample) == hash_chunk(sample)}")
print(f"Разный текст, разный хеш: {hash_chunk(sample) != hash_chunk('Dog.')}")

def dedup_chunks(chunks):
    """Удаление дубликатов чанков с использованием хеш-отпечатков."""
    seen = set()    # Отслеживание увиденных хешей (быстрый поиск!)
    unique = []     # Сохранение только уникальных чанков
    
    for c in chunks:
        h = hash_chunk(c)
        if h in seen:
            continue        # Пропуск дубликата
        seen.add(h)         # Запоминание этого хеша
        unique.append(c)    # Сохранение чанка
    
    return unique

# Добавление дубликата для проверки работы дедупликации
test_docs = cleaned_docs + [cleaned_docs[0]]  # Добавление копии первого документа
deduped = dedup_chunks(test_docs)
print(f'\nДо дедупликации: {len(test_docs)} документов')
print(f'После дедупликации:  {len(deduped)} документов')

## Разделение train/val/test (по документам)

Сохранение связанного текста вместе, избежание утечки данных.

In [ ]:
def split_docs(docs: list, train_p: float = 0.8, val_p: float = 0.1, seed: int = 42):
    """Разделение документов на train/val/test наборы.
    
    Аргументы:
        docs: Список документов для разделения
        train_p: Доля для обучения (по умолчанию 0.8 = 80%)
        val_p: Доля для валидации (по умолчанию 0.1 = 10%)
        seed: Начальное значение для воспроизводимости
    
    Возвращает:
        train, val, test списки (test получает оставшуюся долю)
    """
    # Работа с копией, чтобы не изменять список вызывающего
    docs = list(docs)
    random.seed(seed)
    random.shuffle(docs)
    
    n = len(docs)
    n_train = int(n * train_p)
    n_val = int(n * val_p)
    
    train = docs[:n_train]
    val = docs[n_train:n_train + n_val]
    test = docs[n_train + n_val:]
    
    # Округление может обделить разделение на крошечных корпусах (3 документа
    # при 0.34/0.33 дают int(3*0.33)=0 для val). Если разделение оказалось
    # пустым, заимствуем один документ из самого большого разделения, чтобы
    # каждое получило хотя бы один, когда это возможно.
    if n >= 3:
        for target in (train, val, test):
            if not target:
                donors = sorted((train, val, test), key=len, reverse=True)
                if len(donors[0]) > 1:
                    target.append(donors[0].pop())

    return train, val, test

# Применение разделения к дедуплицированным документам
train_docs, val_docs, test_docs = split_docs(deduped, train_p=0.6, val_p=0.2, seed=42)
print(f'Train: {len(train_docs)}, Val: {len(val_docs)}, Test: {len(test_docs)}')


## Чанкинг для контекстных окон
Разбиение длинного текста с перекрытием для сохранения контекста на границах чанков.

In [ ]:
def chunk_text(text: str, max_chars: int = 200, overlap: int = 50):
    """Разбиение текста на перекрывающиеся чанки.
    
    Аргументы:
        text: Входной текст для разбиения
        max_chars: Максимум символов на чанк
        overlap: Количество символов для перекрытия между чанками
    
    Возвращает:
        Список текстовых чанков
    """
    assert 0 <= overlap < max_chars, "overlap must be smaller than max_chars"

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break  # этот чанк достиг конца; без лишнего хвоста
        start += max_chars - overlap  # продвижение на (max_chars - overlap)
    return chunks

# Применение чанкинга ко всем разделениям
chunked = []
for split, docs in [('train', train_docs), ('val', val_docs), ('test', test_docs)]:
    for d in docs:
        for c in chunk_text(d, max_chars=120, overlap=30):
            chunked.append({'text': c, 'split': split, 'source': 'toy'})
            
print('Всего чанков:', len(chunked))


## Визуализация перекрытия
Посмотрите, как чанки перекрываются для сохранения контекста на границах.

In [ ]:
# Создание тестового текста с четкими позициями
test_text = "A" * 500  # 500 символов
chunks = chunk_text(test_text, max_chars=200, overlap=50)

print(f"Длина текста: {len(test_text)}")
print(f"Количество чанков: {len(chunks)}")
print(f"Длины чанков: {[len(c) for c in chunks]}")

# Проверка перекрытия между последовательными чанками
# Нотация срезов Python:
#   text[-50:]  = последние 50 символов (отрицательный индекс считает с конца)
#   text[:50]   = первые 50 символов
if len(chunks) >= 2:
    # Последние 50 символов чанка 0 должны равняться первым 50 символам чанка 1
    chunk0_end = chunks[0][-50:]    # Последние 50 символов чанка 0
    chunk1_start = chunks[1][:50]   # Первые 50 символов чанка 1
    overlap_matches = chunk0_end == chunk1_start
    
    print(f"\nПроверка перекрытия: {overlap_matches}")
    print(f"Чанк 0 заканчивается: ...{chunks[0][-10:]}")
    print(f"Чанк 1 начинается: {chunks[1][:10]}...")
    
# С реальным текстом
real_text = "This is sentence one. This is sentence two. This is sentence three." * 5
real_chunks = chunk_text(real_text, max_chars=100, overlap=30)
print(f"\nРеальный текст разбит на {len(real_chunks)} частей")
print(f"Чанк 0: ...{real_chunks[0][-40:]}")
print(f"Чанк 1: {real_chunks[1][:40]}...")
print("\n✅ Перекрытие смягчает разрезание предложений на границах чанков (не гарантия).")

## Проверки качества и валидация
Раннее обнаружение проблем с помощью автоматических проверок, которые находят пустые чанки, утечку HTML и проблемы с размерами.

In [ ]:
def sanity_check(chunks, stage_name):
    """Запуск проверок на данных на любом этапе конвейера."""
    print(f"\n{'='*50}")
    print(f"Проверка: {stage_name}")
    print(f"{'='*50}")
    
    if not chunks:
        print("⚠️  ПРЕДУПРЕЖДЕНИЕ: Нет чанков!")
        return
    
    # Базовая статистика
    print(f"✓ Всего чанков: {len(chunks)}")
    lengths = [len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks]
    avg_len = sum(lengths) / len(lengths)
    print(f"✓ Средняя длина: {avg_len:.0f}")
    print(f"✓ Макс длина: {max(lengths)}")
    print(f"✓ Мин длина: {min(lengths)}")
    
    # Проверка проблем
    if max(lengths) > 10 * avg_len:
        print("⚠️  ПРЕДУПРЕЖДЕНИЕ: Макс длина в 10 раз больше средней - чанкинг может быть сломан")
    
    # Проверка утечки HTML
    texts = [c if isinstance(c, str) else c.get('text', '') for c in chunks]
    all_text = ' '.join(texts).lower()
    html_words = {'div', 'span', 'href', 'html', 'class', 'src'}
    found_html = [w for w in html_words if w in all_text]
    if found_html:
        print(f"⚠️  ПРЕДУПРЕЖДЕНИЕ: Найдены HTML-теги: {found_html}")
    else:
        print("✓ Утечка HTML не обнаружена")
    
    # Проверка пустых чанков
    empty = sum(1 for l in lengths if l < 10)
    if empty > 0:
        print(f"⚠️  ПРЕДУПРЕЖДЕНИЕ: {empty} чанков < 10 символов")
    else:
        print("✓ Нет пустых чанков")
    
    # Пример
    sample = texts[0] if texts else "N/A"
    print(f"\n✓ Пример: {sample[:100]}...")
    print()

# Запуск проверок на наших данных после чанкинга
sanity_check(chunked, "После чанкинга")

# Вы можете запускать это после каждого этапа:
# sanity_check(cleaned_docs, "После очистки")
# sanity_check(deduped, "После дедупликации")


## Рабочий пример: Конвейер от начала до конца
Полное прохождение от исходных документов (с HTML и дубликатами) до данных, готовых для JSONL.

In [ ]:
print("="*60)
print("ПОЛНОЕ ПРОХОЖДЕНИЕ КОНВЕЙЕРА ДАННЫХ")
print("="*60)

# Шаг 1: Начало с исходных документов (неопрятных, с дубликатами и HTML)
print("\n📥 ШАГ 1: Исходные документы")
raw_pipeline_docs = [
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc1"},
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc2"},  # точный дубликат!
    {"text": "<p>The dog    ran\tin the park.</p>", "source": "doc3"},
    {"text": "The bird flew over the house.", "source": "doc4"}
]
print(f"   Исходных документов: {len(raw_pipeline_docs)}")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text'][:50]}...")

# Шаг 2: Очистка каждого документа
print("\n🧹 ШАГ 2: Очистка")
for doc in raw_pipeline_docs:
    doc["text"] = clean_text(doc["text"])
print("   HTML удален, пробелы нормализованы")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text']}")

# Сохраняем первый документ с каждым уникальным текстом вместе с источником.
unique_pipeline, seen = [], set()
for doc in raw_pipeline_docs:
    key = hash_chunk(doc['text'])
    if key not in seen:
        seen.add(key)
        unique_pipeline.append(doc)

splits = split_docs(unique_pipeline, train_p=0.34, val_p=0.33, seed=42)
final_records = []
for split, docs in zip(('train', 'val', 'test'), splits):
    print(split, [doc['source'] for doc in docs])
    for doc in docs:
        for chunk in chunk_text(doc['text'], max_chars=50, overlap=10):
            final_records.append(dict(text=chunk, source=doc['source'], split=split, length=len(chunk)))
print(final_records[0])
# Три записи показывают формат, а не достаточный для обучения объём данных.

In [ ]:
def save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(final_records, 'train.jsonl')
print('Записано train.jsonl с', len(final_records), 'записями')

## Быстрая статистика

Процент дубликатов, топ слова и примеры записей.

In [ ]:
def duplicate_rate(texts):
    hashes = [hash_chunk(t) for t in texts]
    return 1 - (len(set(hashes)) / len(hashes))

def top_words(texts, k=10):
    words = " ".join(texts).lower().split()
    return Counter(words).most_common(k)

texts_all = [r['text'] for r in chunked]
print('Процент дубликатов:', duplicate_rate(texts_all))
print('Топ слова:', top_words(texts_all, k=8))
print('Примеры записей:', chunked[:2])

## Резюме

В этом блокноте вы изучили полный конвейер подготовки данных для обучения LLM:

**1. Очистка текста:** Удаление HTML, нормализация пробелов, обработка специальных символов
**2. Дедупликация:** Использование хеширования для идентификации и удаления точных дубликатов
**3. Разделение Train/Val/Test:** Разделение данных на уровне документов для предотвращения утечки
**4. Чанкинг с перекрытием:** Разбиение длинных текстов на части размером с LLM с сохранением контекста
**5. Проверки качества:** Автоматические проверки обнаруживают проблемы на ранних этапах
**6. Формат JSONL:** Сохранение данных в формате, удобном для потоковой передачи

**Ключевые концепции:**
- **Перекрытие** сохраняет контекст на границах чанков (смягчает разрезание предложений пополам; не гарантия)
- **Хеширование** обеспечивает быстрые, детерминированные отпечатки для дедупликации
- **Разделение на уровне документов** сохраняет связанные чанки вместе в одном разделении
- **Проверки качества** обнаруживают утечку HTML, пустые чанки и аномалии размеров до того, как они вызовут проблемы при обучении

**Следующие шаги:**
- Глава 8: Токенизация (преобразование текста → в числа)
- Масштабирование до реальных наборов данных (Wikipedia, Common Crawl, книги)
- Эксперименты с различными значениями перекрытия для вашего случая использования

Конвейер данных — это основа каждой великой LLM!